In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
import geopandas as gpd
import pandas as pd
from config.config import DATA_INTERIM, OUTPUTS


## Spatial Diagnostics

Replicates Figures 1 and 2 from Suss & Oliveira (2023) using 2025 S&S data and 2022-2024 Land Registry Gini values at LSOA level.

Figure 1: Spatial distribution of S&S count by LSOA - London, 2025
Figure 2: Spatial distribution of housing value inequality (Gini) by LSOA - London, 2025

Moran's I tests on OLS and SDM residuals are performed in the modelling notebook (notebooks/05_modelling.ipynb) following the paper's analytic strategy.

In [ ]:
os.makedirs(OUTPUTS / "figures", exist_ok=True)

# Load LSOA boundaries
london_gdf = gpd.read_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021"
)[["LSOA21CD", "geometry"]]

# Load analytical dataset
analytical = pd.read_csv(DATA_INTERIM / "analytical_dataset.csv")

# Merge onto boundaries
london_gdf = london_gdf.merge(
    analytical[["LSOA21CD", "ss_count", "gini_housing"]],
    on="LSOA21CD",
    how="left"
)

print(f"LSOAs loaded: {len(london_gdf)}")
print(f"Null ss_count: {london_gdf['ss_count'].isna().sum()}")
print(f"Null gini_housing: {london_gdf['gini_housing'].isna().sum()}")


## Figure 1 - S&S count by LSOA

Uses the same classification breaks as the paper:
0, 1-10, 11-25,  26-99, 100-199, 200+
Colour scheme: YlGnBu sequential palette (light yellow → teal → dark blue-black)

In [ ]:
# Classify ss_count into paper's break categories
bins = [-1, 0, 10, 25, 99, 199, london_gdf["ss_count"].max()]
labels = ["0", "1–10", "11–25", "26–99", "100–199", "200+"]
london_gdf["ss_class"] = pd.cut(
    london_gdf["ss_count"],
    bins=bins,
    labels=labels
)

# Colour palette matching paper Figure 1 — dark sequential teal/blue-black
palette = ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494", "#0a0a2a"]
color_map = dict(zip(labels, palette))

london_gdf["ss_colour"] = london_gdf["ss_class"].map(color_map)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

london_gdf.plot(
    ax=ax,
    color=london_gdf["ss_colour"].fillna("#cccccc"),
    linewidth=0.1,
    edgecolor="white"
)

# Legend
legend_patches = [
    mpatches.Patch(color=palette[i], label=labels[i])
    for i in range(len(labels))
]
ax.legend(
    handles=legend_patches,
    title="Stops",
    loc="lower right",
    frameon=True,
    fontsize=10,
    title_fontsize=11
)

ax.set_axis_off()
ax.set_title(
    "Number of Stop and Searches — London, 2025, LSOA level",
    fontsize=13,
    pad=12
)

plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "fig1_ss_count.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig1_ss_count.png")


## Figure 2 - Housing value inequality (Gini) by LSOA

Continuous scale from min to max Gini value.
Colour scheme: RdPu sequential palette (light pink → dark purple-black)
LSOAs below the n=30 transaction threshold are shown in grey (no Gini value)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Grey base layer for LSOAs with no Gini value (below threshold)
london_gdf.plot(
    ax=ax,
    color="#cccccc",
    linewidth=0.1,
    edgecolor="white"
)

# Colour LSOAs with Gini values
london_gdf[london_gdf["gini_housing"].notna()].plot(
    ax=ax,
    column="gini_housing",
    cmap="RdPu",
    linewidth=0.1,
    edgecolor="white",
    vmin=london_gdf["gini_housing"].quantile(0.01),
    vmax=london_gdf["gini_housing"].quantile(0.99),
    legend=True,
    legend_kwds={
        "label": "Gini coefficient",
        "orientation": "vertical",
        "shrink": 0.5,
        "pad": 0.02
    }
)

ax.set_axis_off()
ax.set_title(
    "Housing Value Inequality — London, 2025, LSOA level",
    fontsize=13,
    pad=12
)

# Add note for grey areas
ax.annotate(
    "Grey: fewer than 30 transactions (Gini not calculated)",
    xy=(0.01, 0.01),
    xycoords="axes fraction",
    fontsize=8,
    color="#666666"
)

plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "fig2_gini.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig2_gini.png")


## Combined figure - Figures 1 and 2 side by side

In [ ]:
fig = plt.figure(figsize=(22, 10))
gs = GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 0.05], wspace=0.05)

ax_left = fig.add_subplot(gs[0])
ax_right = fig.add_subplot(gs[1])
ax_cbar = fig.add_subplot(gs[2])

# --- Left: S&S count ---
london_gdf.plot(
    ax=ax_left,
    color=london_gdf["ss_colour"].fillna("#cccccc"),
    linewidth=0.1,
    edgecolor="white"
)

legend_patches = [
    mpatches.Patch(color=palette[i], label=labels[i])
    for i in range(len(labels))
]
ax_left.legend(
    handles=legend_patches,
    title="Stops",
    loc="lower left",
    frameon=True,
    fontsize=9,
    title_fontsize=10,
    bbox_to_anchor=(0.01, 0.01),
    borderaxespad=0
)
ax_left.set_axis_off()

# --- Right: Gini ---
london_gdf.plot(
    ax=ax_right,
    color="#cccccc",
    linewidth=0.1,
    edgecolor="white"
)

vmin = london_gdf["gini_housing"].quantile(0.01)
vmax = london_gdf["gini_housing"].quantile(0.99)

london_gdf[london_gdf["gini_housing"].notna()].plot(
    ax=ax_right,
    column="gini_housing",
    cmap="RdPu",
    linewidth=0.1,
    edgecolor="white",
    vmin=vmin,
    vmax=vmax,
    legend=False
)

ax_right.set_axis_off()
ax_right.annotate(
    "Grey: fewer than 30 transactions (Gini not calculated)",
    xy=(0.01, 0.01),
    xycoords="axes fraction",
    fontsize=8,
    color="#666666"
)

# --- Colourbar in dedicated narrow axes ---
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
sm = cm.ScalarMappable(cmap="RdPu", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=ax_cbar)
cbar.set_label("Gini coefficient", fontsize=10)

# Fixed-position titles so both are at identical vertical positions
# Get axes bounding boxes to align titles
fig.canvas.draw()
left_bbox = ax_left.get_position()
right_bbox = ax_right.get_position()
title_y = max(left_bbox.y1, right_bbox.y1) + 0.02

fig.text(
    (left_bbox.x0 + left_bbox.x1) / 2, title_y,
    "Number of Stop and Searches\nLondon, 2025, LSOA level",
    ha="center", va="bottom", fontsize=12
)
fig.text(
    (right_bbox.x0 + right_bbox.x1) / 2, title_y,
    "Housing Value Inequality\nLondon, 2025, LSOA level",
    ha="center", va="bottom", fontsize=12
)

plt.suptitle(
    "Spatial Distribution of Stop and Search and Housing Inequality — London, 2025",
    fontsize=14,
    y=title_y + 0.11
)

plt.savefig(
    OUTPUTS / "figures" / "fig1_fig2_combined.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("Saved fig1_fig2_combined.png")